# Jaw Clench Validation Report

This notebook visualizes the latest NeuroPawn jaw-clench validation run. It uses EEG channel 2 only, highlights guided jaw-clench labels, and plots validation predictions against the selected threshold.

In [ ]:
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np

from neuro_cursor.config import load_config
from neuro_cursor.jaw_features import FEATURE_NAMES, interval_labels, jaw_signal, read_labels

ROOT = Path.cwd()
REPORT_DIR = max((ROOT / 'reports' / 'jaw_eval').iterdir(), key=lambda p: p.stat().st_mtime)
METRICS_PATH = REPORT_DIR / 'metrics.json'
TIMELINE_PATH = REPORT_DIR / 'prediction_timeline.csv'
SWEEP_PATH = REPORT_DIR / 'threshold_sweep.csv'

metrics = json.loads(METRICS_PATH.read_text())
config = load_config(ROOT / 'config' / 'neuro_cursor.yaml')
print('Report:', REPORT_DIR)
print('Model:', metrics['profile_dir'])

## Validation Summary

In [ ]:
summary = {
    'status': metrics['validation_status'],
    'threshold': metrics['selected_threshold'],
    **metrics['metrics'],
}
for key, value in summary.items():
    print(f'{key:28s}: {value}')

print('\nTraining sessions:')
for session in metrics['metadata']['train_sessions']:
    print('  ', session)
print('\nValidation sessions:')
for session in metrics['metadata']['validation_sessions']:
    print('  ', session)

## Raw EEG 2 With Guided Labels

Orange spans are labeled `jaw_clench` intervals. The trace is median-centered per session for readability.

In [ ]:
def load_raw(session_dir):
    with np.load(session_dir / 'raw.npz') as archive:
        return np.asarray(archive['raw'], dtype=float)

def plot_session_trace(ax, session_path, title):
    raw = load_raw(session_path)
    labels = read_labels(session_path / 'labels.jsonl')
    signal = jaw_signal(raw, config.jaw)
    t = np.arange(signal.size) / config.jaw.sampling_rate
    centered = signal - np.median(signal)
    ax.plot(t, centered, color='#cfd4d8', linewidth=0.8)
    for label in interval_labels(labels):
        if label.get('label') == 'jaw_clench':
            start = int(label['start_sample']) / config.jaw.sampling_rate
            end = int(label['end_sample']) / config.jaw.sampling_rate
            ax.axvspan(start, end, color='#f0b84f', alpha=0.28)
    ax.set_title(title, loc='left', fontsize=10)
    ax.set_ylabel('EEG2 centered')
    ax.grid(alpha=0.18)

sessions = [Path(p) for p in metrics['metadata']['train_sessions'] + metrics['metadata']['validation_sessions']]
fig, axes = plt.subplots(len(sessions), 1, figsize=(14, 2.2 * len(sessions)), sharex=False)
if len(sessions) == 1:
    axes = [axes]
for ax, session in zip(axes, sessions):
    role = 'validation' if str(session) in metrics['metadata']['validation_sessions'] else 'train'
    plot_session_trace(ax, ROOT / session, f'{session.name} ({role})')
axes[-1].set_xlabel('seconds from session start')
fig.tight_layout()

## Validation Predictions

Each point is one validation window. Orange points are true clench windows; gray points are relaxed windows.

In [ ]:
timeline = []
with TIMELINE_PATH.open(newline='') as handle:
    for row in csv.DictReader(handle):
        row['true_label'] = int(row['true_label'])
        row['predicted_label'] = int(row['predicted_label'])
        row['probability'] = float(row['probability'])
        row['start_sample'] = int(row['start_sample'])
        row['end_sample'] = int(row['end_sample'])
        timeline.append(row)

x = np.arange(len(timeline))
prob = np.array([row['probability'] for row in timeline])
truth = np.array([row['true_label'] for row in timeline])
pred = np.array([row['predicted_label'] for row in timeline])
threshold = metrics['selected_threshold']

fig, ax = plt.subplots(figsize=(14, 4))
ax.scatter(x[truth == 0], prob[truth == 0], s=28, color='#8a949b', label='relaxed truth')
ax.scatter(x[truth == 1], prob[truth == 1], s=48, color='#f0b84f', label='clench truth')
ax.axhline(threshold, color='#e15b64', linestyle='--', linewidth=1.4, label=f'threshold {threshold:.2f}')
ax.set_ylim(-0.04, 1.04)
ax.set_xlabel('validation window index')
ax.set_ylabel('jaw_clench probability')
ax.grid(alpha=0.18)
ax.legend(loc='upper right')
fig.tight_layout()

print('false positives:', int(((pred == 1) & (truth == 0)).sum()))
print('false negatives:', int(((pred == 0) & (truth == 1)).sum()))

## Threshold Sweep

The selected threshold favors low false positives for cursor safety.

In [ ]:
sweep = []
with SWEEP_PATH.open(newline='') as handle:
    for row in csv.DictReader(handle):
        sweep.append({key: float(value) for key, value in row.items()})

thr = np.array([row['threshold'] for row in sweep])
precision = np.array([row['precision'] for row in sweep])
recall = np.array([row['recall'] for row in sweep])
fp_per_min = np.array([row['false_positives_per_minute'] for row in sweep])

fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(thr, precision, label='precision', color='#7bc96f')
ax1.plot(thr, recall, label='recall', color='#4aa3df')
ax1.axvline(threshold, color='#e15b64', linestyle='--', linewidth=1.4)
ax1.set_xlabel('threshold')
ax1.set_ylabel('precision / recall')
ax1.set_ylim(-0.04, 1.04)
ax1.grid(alpha=0.18)

ax2 = ax1.twinx()
ax2.plot(thr, fp_per_min, label='false positives/min', color='#f0b84f', alpha=0.8)
ax2.set_ylabel('false positives per minute')

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='center right')
fig.tight_layout()

## Feature Separation

Higher robust separation means the feature differs more between relaxed and clench windows in validation.

In [ ]:
feature_rows = metrics['feature_separation']['features']
names = list(feature_rows.keys())
separation = np.array([feature_rows[name]['robust_separation'] for name in names])
order = np.argsort(separation)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(np.array(names)[order], separation[order], color='#7bc96f')
ax.axvline(1.0, color='#e15b64', linestyle='--', linewidth=1.2, label='usable separation')
ax.set_xlabel('robust separation')
ax.grid(axis='x', alpha=0.18)
ax.legend(loc='lower right')
fig.tight_layout()

## Data Quality

In [ ]:
for split in ('train', 'validation'):
    print(split.upper())
    for row in metrics['data_quality'][split]:
        print(row['session'])
        print('  samples:', row['samples'])
        print('  neutral intervals:', row['neutral_intervals'])
        print('  jaw clench intervals:', row['jaw_clench_intervals'])
        print('  timestamp rate:', f"{row['timestamp_rate_hz']:.1f} Hz")
        if row['warnings']:
            print('  warnings:', '; '.join(row['warnings']))
        if row['errors']:
            print('  errors:', '; '.join(row['errors']))